# Additional What-if Analysis

Run **all cells** in this notebook. The first code cell automatically executes the supplied `model.ipynb`, then the two analyses below use its model, inputs, and optimized weights. No additional visualization is included.

Keep both notebooks and the three input CSV files in the same working directory. The baseline requires the same Python packages and a working Gurobi license as the original project. Existing baseline output is captured rather than displayed here; errors still stop execution. The baseline may still save its original figure files as part of its existing code.

A. Exclude ESLT, CAH, or both at portfolio formation and reoptimize.

B. Increase every stock's volatility by 25%, retaining the original portfolio weights, expected returns, and correlations. No reoptimization is performed.

In [1]:
# Automatically execute model.ipynb before running either what-if scenario.
# Capture its existing output so this notebook shows only the new analysis.
import json
from pathlib import Path
from IPython.utils.io import capture_output

_model_path = Path("model.ipynb")
if not _model_path.is_file():
    raise FileNotFoundError("Place model.ipynb and this notebook in the same working directory.")

def _execute_baseline_notebook(path, namespace):
    notebook = json.loads(path.read_text(encoding="utf-8"))
    # The supplied model notebook contains ordinary Python code cells.
    with capture_output(stdout=True, stderr=True, display=True):
        for cell_number, model_cell in enumerate(notebook["cells"], start=1):
            if model_cell["cell_type"] == "code":
                code = "".join(model_cell["source"])
                exec(compile(code, f"{path.name}:cell_{cell_number}", "exec"), namespace)
    namespace["plt"].close("all")

_execute_baseline_notebook(_model_path, globals())
print("model.ipynb executed successfully. Baseline variables are ready.")
print("The following scenarios use the baseline model and original optimized weights.")

model.ipynb executed successfully. Baseline variables are ready.
The following scenarios use the baseline model and original optimized weights.


#### What if key holdings are unavailable at portfolio formation?

We exclude ESLT, CAH, and then both. Each case reoptimizes a copy of the baseline model with its existing return floor, 20% weight cap, and 0.40 correlation threshold. We compare monthly Normal 95% CVaR and realized January–March buy-and-hold returns. The latter are supplemental observations over three months.

In [2]:
# Use the original baseline weights saved in the Results section.
baseline_weights = portfolio["Optimized Weight"].to_numpy(dtype=float, copy=True)
stock_growth = np.prod(1 + stock_returns, axis=1)

def summarize_whatif(label, scenario_weights, covariance):
    expected = float(mu @ scenario_weights)
    volatility = float(np.sqrt(scenario_weights @ covariance @ scenario_weights))
    return {
        "Scenario": label,
        "Expected Monthly Return": expected,
        "Monthly Std Dev": volatility,
        "Monthly 95% CVaR": -expected + k * volatility,
        "Holdings": int(np.count_nonzero(scenario_weights > 1e-6)),
    }

baseline_metrics = summarize_whatif("Baseline", baseline_weights, Sigma)
exclusion_rows = [{
    **baseline_metrics,
    "Status": "Optimal",
    "Jan–Mar Buy-and-Hold Return": float(baseline_weights @ stock_growth - 1),
}]
scenario_allocations = {"Baseline": baseline_weights}
exclusions = [("ESLT",), ("CAH",), ("ESLT", "CAH")]

for excluded in exclusions:
    label = "Exclude " + " and ".join(excluded)
    with m.copy() as scenario_model:
        scenario_model.Params.OutputFlag = 0
        scenario_model.Params.MIPGap = 1e-7
        scenario_vars = [scenario_model.getVarByName(f"w[{i}]") for i in range(len(tickers))]
        for ticker in excluded:
            scenario_vars[tickers.index(ticker)].UB = 0.0
        scenario_model.optimize()
        if scenario_model.Status == GRB.OPTIMAL:
            scenario_weights = np.array([variable.X for variable in scenario_vars])
            scenario_allocations[label] = scenario_weights
            exclusion_rows.append({
                **summarize_whatif(label, scenario_weights, Sigma),
                "Status": "Optimal",
                "Jan–Mar Buy-and-Hold Return": float(scenario_weights @ stock_growth - 1),
            })
        else:
            status = "Infeasible" if scenario_model.Status == GRB.INFEASIBLE else f"Gurobi status {scenario_model.Status}"
            exclusion_rows.append({"Scenario": label, "Status": status})

whatif_exclusions = pd.DataFrame(exclusion_rows).set_index("Scenario")
whatif_exclusions["CVaR Change (pp)"] = 100 * (
    whatif_exclusions["Monthly 95% CVaR"] - baseline_metrics["Monthly 95% CVaR"]
)
whatif_exclusion_weights = pd.DataFrame(scenario_allocations, index=pd.Index(tickers, name="Ticker"))
print(whatif_exclusions.to_string(formatters={
    column: "{:.4%}".format for column in
    ["Expected Monthly Return", "Monthly Std Dev", "Monthly 95% CVaR", "Jan–Mar Buy-and-Hold Return"]
}))
print("\nPortfolio weights (%):")
print((100 * whatif_exclusion_weights.loc[whatif_exclusion_weights.gt(1e-6).any(axis=1)]).round(3))

                     Expected Monthly Return Monthly Std Dev Monthly 95% CVaR  Holdings   Status Jan–Mar Buy-and-Hold Return  CVaR Change (pp)
Scenario                                                                                                                                      
Baseline                             3.0046%         4.3315%          5.9301%         9  Optimal                    10.5580%          0.000000
Exclude ESLT                         2.9714%         4.7169%          6.7582%         9  Optimal                     1.4734%          0.828030
Exclude CAH                          3.0767%         4.6359%          6.4859%         9  Optimal                    11.0786%          0.555814
Exclude ESLT and CAH                 3.0649%         5.1863%          7.6330%         9  Optimal                     1.3799%          1.702861

Portfolio weights (%):
        Baseline  Exclude ESLT  Exclude CAH  Exclude ESLT and CAH
Ticker                                              

#### What if all stocks become 25% more volatile?

We multiply each stock's standard deviation by 1.25 while preserving expected returns and correlations, so the covariance matrix is multiplied by $1.25^2$. This is a stress test of the original portfolio at its existing weights. We do not reoptimize or alter the historical realized returns.

In [3]:
volatility_multiplier = 1.25
stressed_covariance = volatility_multiplier**2 * Sigma
whatif_volatility = pd.DataFrame([
    baseline_metrics,
    summarize_whatif("Stock volatility +25%", baseline_weights, stressed_covariance),
]).set_index("Scenario")
whatif_volatility["CVaR Change (pp)"] = 100 * (
    whatif_volatility["Monthly 95% CVaR"] - baseline_metrics["Monthly 95% CVaR"]
)
print(whatif_volatility.to_string(formatters={
    column: "{:.4%}".format for column in
    ["Expected Monthly Return", "Monthly Std Dev", "Monthly 95% CVaR"]
}))

                      Expected Monthly Return Monthly Std Dev Monthly 95% CVaR  Holdings  CVaR Change (pp)
Scenario                                                                                                  
Baseline                              3.0046%         4.3315%          5.9301%         9          0.000000
Stock volatility +25%                 3.0046%         5.4144%          8.1638%         9          2.233676


**Interpretation.** Scenario A compares portfolios reoptimized at formation; their January–March returns use the existing buy-and-hold evaluation. Scenario B is a fixed-weight risk calculation, with the covariance matrix multiplied by $1.25^2$; it does not simulate a three-month trading policy or change historical realized returns. All reported monthly CVaR estimates use the Normal assumption.